In [6]:
import os
import json
import random
import pandas as pd
import numpy as np
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

TARGET_FOLDER = os.path.join('osmnx_inputs')

# ═══════════════════════════════════════════════════════════════════════
# BAGIAN 1 — ACO
# ═══════════════════════════════════════════════════════════════════════

def hitung_fitness_aco(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user):
    w_jarak = 10.0
    w_jalan = 5.0
    w_penalti_cluster = 50.0
    w_hari = 30.0

    fitness_total = len(solusi_multi_hari) * w_hari
    log_itinerary = []

    for idx_hari, rute in enumerate(solusi_multi_hari, start=1):
        if idx_hari % 2 != 0:
            nama_hari = f"Pekan {(idx_hari+1)//2} - Sabtu (Jalan Berat)"
            w_penalti_fatigue = 1.5
            w_penalti_jarak   = 5.0
        else:
            nama_hari = f"Pekan {idx_hari//2} - Minggu (Recovery)"
            w_penalti_fatigue = 8.0
            w_penalti_jarak   = 25.0

        jarak_m = fatigue = skor_jalan = p_cluster = 0.0

        for k in range(len(rute) - 1):
            a, t = rute[k], rute[k+1]
            dm = dist_np[a, t]
            dz = elev_np[a, t]
            js = road_np[a, t]

            jarak_m   += dm
            skor_jalan += js

            if dz > 0 and dm > 0:
                fatigue += dm * ((dz / dm) ** 2) * 400.0
            else:
                fatigue += dm * 0.0001

            if a != 0 and t != 0:
                if cluster_np[a-1] != cluster_np[t-1]:
                    p_cluster += w_penalti_cluster

        jkm      = jarak_m / 1000.0
        p_jarak  = max(0.0, jkm - max_km_user) * w_penalti_jarak
        p_fatigue = fatigue * w_penalti_fatigue

        fitness_hari = (jkm * w_jarak) + (skor_jalan * w_jalan) + p_jarak + p_fatigue + p_cluster
        fitness_total += fitness_hari

        log_itinerary.append({
            "Hari/Trip": nama_hari,
            "Jarak (Km)": round(jkm, 2),
            "Fatigue Index": round(fatigue, 2),
            "Skor Jalan OSMnx": round(skor_jalan, 2),
            "P_Fatigue": round(p_fatigue, 2),
            "P_Cluster": round(p_cluster, 2),
        })

    return fitness_total, log_itinerary


def run_aco(dist_np, elev_np, road_np, cluster_np,
            alpha, beta, evaporation, max_km_user, banned_nodes,
            num_ants=15, iterations=30):

    num_nodes = len(dist_np)
    max_m     = max_km_user * 1000
    pheromone = np.full((num_nodes, num_nodes), 0.1)

    max_dist = dist_np.max()
    max_elev = elev_np.max()
    if max_elev <= 0:
        max_elev = 1.0

    best_score = float('inf')
    best_route = []
    best_log   = None

    for _ in range(iterations):
        list_solusi  = []
        list_fitness = []
        list_log     = []

        for _ in range(num_ants):
            destinasi_tersisa = [i for i in range(1, num_nodes) if i not in banned_nodes]
            rute_multi_hari   = []

            while destinasi_tersisa:
                rute_hari_ini = [0]
                jarak_hari    = 0.0

                while destinasi_tersisa:
                    curr = rute_hari_ini[-1]
                    valid_candidates = [
                        c for c in destinasi_tersisa
                        if jarak_hari + dist_np[curr, c] + dist_np[c, 0] <= max_m
                    ]

                    if not valid_candidates:
                        break

                    probs = []
                    for cand in valid_candidates:
                        tau    = pheromone[curr][cand] ** alpha
                        d_norm = dist_np[curr, cand] / max_dist
                        r_norm = road_np[curr, cand] / 5.0

                        cluster_norm = 0.0
                        if curr != 0 and cand != 0:
                            if cluster_np[curr-1] != cluster_np[cand-1]:
                                cluster_norm = 0.15

                        if len(rute_multi_hari) % 2 == 0:
                            eta = 1.0 / (0.8 * d_norm + 0.2 * r_norm + cluster_norm + 0.001)
                        else:
                            dz     = elev_np[curr, cand]
                            f_norm = max(0, dz) / (max_elev + 1e-6)
                            eta    = 1.0 / (0.5 * d_norm + 0.2 * r_norm + 0.3 * f_norm + cluster_norm + 0.001)

                        probs.append(tau * (eta ** beta))

                    sum_p  = sum(probs)
                    next_c = (random.choice(valid_candidates) if sum_p == 0
                              else random.choices(valid_candidates, weights=[p/sum_p for p in probs], k=1)[0])

                    rute_hari_ini.append(next_c)
                    jarak_hari += dist_np[curr, next_c]
                    destinasi_tersisa.remove(next_c)

                rute_hari_ini.append(0)

                if len(rute_hari_ini) == 2:
                    raise ValueError(
                        f"Infeasible: destinasi dengan jarak PP > {max_km_user} km lolos filter."
                    )

                rute_multi_hari.append(rute_hari_ini)

            f_score, lg = hitung_fitness_aco(
                rute_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user
            )
            list_solusi.append(rute_multi_hari)
            list_fitness.append(f_score)
            list_log.append(lg)

            if f_score < best_score:
                best_score = f_score
                best_route = rute_multi_hari
                best_log   = lg

        pheromone *= (1.0 - evaporation)
        for idx, solusi in enumerate(list_solusi):
            fit_val = list_fitness[idx]
            if fit_val > 0:
                deposit = 2000 / fit_val
                for rute in solusi:
                    for k in range(len(rute) - 1):
                        pheromone[rute[k]][rute[k+1]] += deposit

    return best_score, best_route, best_log




In [7]:
# ═══════════════════════════════════════════════════════════════════════
# BAGIAN 2 — GA
# ═══════════════════════════════════════════════════════════════════════

def hitung_fitness_ga(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user):
    # Fungsi fitness identik dengan ACO — satu sumber kebenaran
    return hitung_fitness_aco(solusi_multi_hari, dist_np, elev_np, road_np, cluster_np, max_km_user)


def buat_kromosom(num_nodes, banned_nodes):
    k = [i for i in range(1, num_nodes) if i not in banned_nodes]
    random.shuffle(k)
    return k


def decode_kromosom(kromo, dist_np, max_km_user):
    max_m = max_km_user * 1000
    hasil = []
    sisa  = list(kromo)

    while sisa:
        rute  = [0]
        jarak = 0.0
        while sisa:
            pos  = rute[-1]
            tuj  = sisa[0]
            dtuj = dist_np[pos, tuj]
            if (jarak + dtuj > max_m) and (random.random() > 0.15):
                break
            rute.append(tuj)
            jarak += dtuj
            sisa.pop(0)
        rute.append(0)
        hasil.append(rute)

    return hasil


def order_crossover(p1, p2):
    n    = len(p1)
    a, b = sorted(random.sample(range(n), 2))
    anak = [None] * n
    anak[a:b+1] = p1[a:b+1]
    sisa = [g for g in p2 if g not in anak[a:b+1]]
    ptr  = 0
    for i in range(n):
        if anak[i] is None:
            anak[i] = sisa[ptr]
            ptr += 1
    return anak


def swap_mutation(k, prob):
    k = k[:]
    if random.random() < prob:
        i, j    = random.sample(range(len(k)), 2)
        k[i], k[j] = k[j], k[i]
    return k


def tournament_selection(pop, fit, k):
    idx = random.sample(range(len(pop)), k)
    return pop[min(idx, key=lambda i: fit[i])][:]


def run_ga(dist_np, elev_np, road_np, cluster_np,
           pop_size, prob_cross, prob_mut, tournament_k,
           max_km_user, banned_nodes, iterations=100):

    num_nodes = len(dist_np)
    populasi  = [buat_kromosom(num_nodes, banned_nodes) for _ in range(pop_size)]

    best_score = float('inf')
    best_route = []
    best_log   = None

    for _ in range(iterations):
        fit_list = []
        log_list = []

        for k in populasi:
            rute    = decode_kromosom(k, dist_np, max_km_user)
            f_score, lg = hitung_fitness_ga(rute, dist_np, elev_np, road_np, cluster_np, max_km_user)
            fit_list.append(f_score)
            log_list.append(lg)

        idx_best = int(np.argmin(fit_list))
        if fit_list[idx_best] < best_score:
            best_score = fit_list[idx_best]
            best_route = decode_kromosom(populasi[idx_best], dist_np, max_km_user)
            best_log   = log_list[idx_best]

        sorted_idx   = np.argsort(fit_list)
        elit         = [populasi[i][:] for i in sorted_idx[:2]]
        generasi_baru = elit[:]

        while len(generasi_baru) < pop_size:
            p1 = tournament_selection(populasi, fit_list, tournament_k)
            p2 = tournament_selection(populasi, fit_list, tournament_k)
            if random.random() < prob_cross:
                a1 = order_crossover(p1, p2)
                a2 = order_crossover(p2, p1)
            else:
                a1, a2 = p1[:], p2[:]
            generasi_baru.append(swap_mutation(a1, prob_mut))
            if len(generasi_baru) < pop_size:
                generasi_baru.append(swap_mutation(a2, prob_mut))

        populasi = generasi_baru

    return best_score, best_route, best_log




In [8]:
# ═══════════════════════════════════════════════════════════════════════
# BAGIAN 3 — PRE-FLIGHT CHECK (dipakai ACO & GA, tidak duplikat)
# ═══════════════════════════════════════════════════════════════════════

def preflight_check(dist_np, names):
    """
    Interaktif: tanya batas km, validasi node infeasible, kembalikan
    (MAX_KM_USER, BANNED_NODES).
    """
    BANNED_NODES = []
    MAX_KM_USER  = 20.10

    print("\n" + "=" * 65)
    print("🚲  PRE-FLIGHT SANITY CHECK: KAPASITAS USER")
    print("=" * 65)

    while True:
        try:
            raw = input("\nBatas maksimal jarak per hari (Km) [Enter = 20.10]: ").strip()
            MAX_KM_USER = float(raw) if raw else 20.10
        except ValueError:
            print("❌ Masukkan angka yang valid!")
            continue

        max_m = MAX_KM_USER * 1000
        infeasible = [
            (i, names[i], dist_np[0, i] + dist_np[i, 0])
            for i in range(1, len(names))
            if dist_np[0, i] + dist_np[i, 0] > max_m
        ]

        if infeasible:
            print(f"\n⚠️  {len(infeasible)} destinasi dengan jarak PP > {MAX_KM_USER} km:")
            for idx, nama, jarak in infeasible:
                print(f"   - {nama}  (PP: {round(jarak/1000, 2)} km)")

            opsi = input("\nHapus destinasi tersebut agar model feasible? (y/n): ").strip().lower()
            if opsi == 'y':
                BANNED_NODES = [n[0] for n in infeasible]
                print("✅ Destinasi di-blacklist.")
                break
            else:
                print("❌ Coba masukkan limit yang lebih besar.")
        else:
            BANNED_NODES = []
            print("✅ Semua destinasi lolos uji kapasitas fisik harian!")
            break

    return MAX_KM_USER, BANNED_NODES




In [9]:
# ═══════════════════════════════════════════════════════════════════════
# BAGIAN 4 — DASHBOARD GABUNGAN (satu-satunya HTML output)
# ═══════════════════════════════════════════════════════════════════════

def generate_dashboard_gabungan(
    rute_f_ga, log_f_ga,
    rute_f_aco, log_f_aco,
    names, route_registry,
    target_folder='osmnx_inputs',
    output_filename='dashboard_gabungan.html'
):
    print("\n" + "=" * 65)
    print("🌍  MEMBUAT DASHBOARD GABUNGAN...")

    def _build(rute, log):
        metrik = [
            {
                "hari":    l["Hari/Trip"],
                "jarak":   l["Jarak (Km)"],
                "fatigue": l["Fatigue Index"],
                "jalan":   l["Skor Jalan OSMnx"],
            }
            for l in log
        ]
        rute_js = []
        for hari in rute:
            hari_data = []
            for k in range(len(hari) - 1):
                a, t  = hari[k], hari[k+1]
                key   = f"{a}_{t}"
                if key in route_registry:
                    hari_data.append({
                        "asal_nama":      names[a],
                        "tujuan_nama":    names[t],
                        "is_depot_asal":  (a == 0),
                        "is_depot_tujuan": (t == 0),
                        "jalur":          route_registry[key],
                    })
            rute_js.append(hari_data)
        return json.dumps(rute_js), json.dumps(metrik)

    jrga,  jmga  = _build(rute_f_ga,  log_f_ga)
    jraco, jmaco = _build(rute_f_aco, log_f_aco)

    OUTPUT_PATH = os.path.join(target_folder, output_filename)

    html = f"""<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dashboard Optimasi Rute Sepeda — Surabaya</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=Space+Grotesk:wght@500;700&display=swap" rel="stylesheet">
<style>
*,*::before,*::after{{box-sizing:border-box;margin:0;padding:0}}
:root{{
  --ga:#2563eb;--ga-light:#dbeafe;
  --aco:#16a34a;--aco-light:#dcfce7;
  --ink:#0f172a;--ink-mid:#475569;--ink-faint:#94a3b8;
  --surface:#ffffff;--bg:#f1f5f9;--border:#e2e8f0;
  --radius:10px;--sidebar:380px;--header:52px;
  --shadow:0 1px 3px rgba(0,0,0,.08),0 4px 12px rgba(0,0,0,.06);
}}
html,body{{height:100%;font-family:'Inter',sans-serif;background:var(--bg);color:var(--ink)}}
#shell{{display:flex;flex-direction:column;height:100vh}}
#topbar{{height:var(--header);background:var(--ink);display:flex;align-items:center;gap:16px;padding:0 20px;flex-shrink:0;border-bottom:2px solid #1e293b}}
.brand{{font-family:'Space Grotesk',sans-serif;font-size:15px;font-weight:700;color:#fff;letter-spacing:-.3px;white-space:nowrap}}
.brand span{{color:var(--ga)}}
#tab-bar{{display:flex;gap:4px;background:#1e293b;border-radius:8px;padding:4px;margin-left:auto}}
.tab-btn{{padding:6px 18px;border:none;border-radius:6px;font-family:'Space Grotesk',sans-serif;font-size:13px;font-weight:600;letter-spacing:.3px;cursor:pointer;transition:all .2s;color:var(--ink-faint);background:transparent}}
.tab-btn.ga.active{{background:var(--ga);color:#fff}}
.tab-btn.aco.active{{background:var(--aco);color:#fff}}
.tab-btn:not(.active):hover{{color:#fff;background:#334155}}
.tab-badge{{display:inline-block;margin-left:6px;padding:1px 6px;border-radius:20px;font-size:10px;font-weight:700;vertical-align:middle;background:rgba(255,255,255,.2);color:rgba(255,255,255,.8)}}
#content{{display:flex;flex:1;overflow:hidden}}
#map-wrap{{flex:1;position:relative}}
#map{{height:100%;width:100%}}
#algo-badge{{position:absolute;top:16px;left:16px;z-index:900;padding:6px 14px;border-radius:20px;font-family:'Space Grotesk',sans-serif;font-size:12px;font-weight:700;letter-spacing:.4px;color:#fff;box-shadow:var(--shadow);transition:background .3s}}
#algo-badge.ga{{background:var(--ga)}}
#algo-badge.aco{{background:var(--aco)}}
.elev-legend{{position:absolute;bottom:28px;left:20px;z-index:900;background:var(--surface);padding:12px 14px;border-radius:var(--radius);box-shadow:var(--shadow);min-width:230px}}
.elev-legend-title{{font-size:11px;font-weight:600;color:var(--ink-mid);margin-bottom:6px;text-transform:uppercase;letter-spacing:.5px}}
.grad-bar{{height:10px;border-radius:4px;background:linear-gradient(to right,#22c55e,#facc15,#ef4444)}}
.grad-labels{{display:flex;justify-content:space-between;margin-top:4px;font-size:10px;color:var(--ink-faint)}}
#sidebar{{width:var(--sidebar);height:100%;flex-shrink:0;background:var(--surface);border-left:1px solid var(--border);display:flex;flex-direction:column;overflow:hidden}}
#sidebar-header{{padding:16px 18px 12px;flex-shrink:0;border-bottom:1px solid var(--border)}}
#sidebar-title{{font-family:'Space Grotesk',sans-serif;font-size:15px;font-weight:700;color:var(--ink);margin-bottom:10px}}
#btn-all{{width:100%;padding:9px;border:none;border-radius:7px;font-family:'Space Grotesk',sans-serif;font-size:12px;font-weight:700;letter-spacing:.3px;cursor:pointer;transition:all .2s;color:#fff}}
#btn-all.ga{{background:var(--ga)}}
#btn-all.aco{{background:var(--aco)}}
#btn-all:hover{{filter:brightness(1.1)}}
#stats-strip{{display:flex;gap:8px;padding:10px 18px;background:var(--bg);border-bottom:1px solid var(--border);flex-shrink:0}}
.stat-pill{{flex:1;background:var(--surface);border:1px solid var(--border);border-radius:8px;padding:8px 10px;text-align:center}}
.stat-pill .sv{{font-size:16px;font-weight:700;color:var(--ink)}}
.stat-pill .sl{{font-size:10px;color:var(--ink-faint);margin-top:1px}}
#cards-wrap{{flex:1;overflow-y:auto;padding:14px;display:flex;flex-direction:column;gap:10px}}
.day-card{{border:1.5px solid var(--border);border-radius:var(--radius);padding:13px 14px;cursor:pointer;transition:box-shadow .15s,border-color .15s,transform .15s;background:var(--surface)}}
.day-card:hover{{box-shadow:var(--shadow);transform:translateY(-1px)}}
.day-card.active-ga{{border-color:var(--ga);background:var(--ga-light)}}
.day-card.active-aco{{border-color:var(--aco);background:var(--aco-light)}}
.card-top{{display:flex;align-items:center;gap:8px;margin-bottom:9px}}
.day-badge{{padding:2px 8px;border-radius:20px;font-size:10px;font-weight:700;color:#fff}}
.day-badge.sabtu{{background:#dc2626}}
.day-badge.minggu{{background:#16a34a}}
.card-title{{font-size:13px;font-weight:600;color:var(--ink);line-height:1.3}}
.metrics{{display:grid;grid-template-columns:1fr 1fr 1fr;gap:6px;margin-top:8px}}
.metric-box{{background:var(--bg);border-radius:6px;padding:6px 8px}}
.metric-box .mk{{font-size:10px;color:var(--ink-faint);margin-bottom:2px}}
.metric-box .mv{{font-size:13px;font-weight:700;color:var(--ink)}}
#banned-notice{{margin:0 14px 10px;padding:8px 12px;background:#fef3c7;border:1px solid #fcd34d;border-radius:8px;font-size:11px;color:#92400e;display:none}}
</style>
</head>
<body>
<div id="shell">
  <div id="topbar">
    <div class="brand">🚴 Surabaya<span>Route</span> · Optimasi Rute Sepeda</div>
    <div id="tab-bar">
      <button class="tab-btn ga active" onclick="switchTab('ga')">Genetic Algorithm <span class="tab-badge">GA</span></button>
      <button class="tab-btn aco" onclick="switchTab('aco')">Ant Colony Opt. <span class="tab-badge">ACO</span></button>
    </div>
  </div>
  <div id="content">
    <div id="map-wrap">
      <div id="map"></div>
      <div id="algo-badge" class="ga">Genetic Algorithm</div>
      <div class="elev-legend">
        <div class="elev-legend-title">Elevasi Tanjakan</div>
        <div class="grad-bar"></div>
        <div class="grad-labels"><span>0m (datar)</span><span>landai</span><span>&gt;25m (berat)</span></div>
      </div>
    </div>
    <div id="sidebar">
      <div id="sidebar-header">
        <div id="sidebar-title">Itinerary Multi-Hari — GA</div>
        <button id="btn-all" class="ga" onclick="tampilkanSemua()">Tampilkan Seluruh Rute</button>
      </div>
      <div id="stats-strip">
        <div class="stat-pill"><div class="sv" id="stat-hari">—</div><div class="sl">Total Hari</div></div>
        <div class="stat-pill"><div class="sv" id="stat-km">—</div><div class="sl">Total Km</div></div>
        <div class="stat-pill"><div class="sv" id="stat-fatigue">—</div><div class="sl">Avg Fatigue</div></div>
      </div>
      <div id="banned-notice"></div>
      <div id="cards-wrap"></div>
    </div>
  </div>
</div>
<script>
const DATA = {{
  ga:  {{ rute: {jrga},  metrik: {jmga}  }},
  aco: {{ rute: {jraco}, metrik: {jmaco} }}
}};
const NAMES_TOTAL = {len(names)};

const map = L.map('map').setView([-7.262015, 112.739727], 13);
L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{attribution:'&copy; OpenStreetMap &copy; CARTO'}}).addTo(map);
let layerGroup = L.layerGroup().addTo(map);
let activeTab  = 'ga';

function elevColor(e) {{
  const v = Math.max(0, Math.min(e, 25));
  let r, g, b;
  if (v <= 12.5) {{
    const t = v / 12.5;
    r = Math.round(0x22 + t*(0xfa-0x22));
    g = Math.round(0xc5 + t*(0xcc-0xc5));
    b = Math.round(0x5e + t*(0x15-0x5e));
  }} else {{
    const t = (v-12.5) / 12.5;
    r = Math.round(0xfa + t*(0xef-0xfa));
    g = Math.round(0xcc + t*(0x44-0xcc));
    b = Math.round(0x15 + t*(0x44-0x15));
  }}
  return `rgb(${{r}},${{g}},${{b}})`;
}}

function renderMap(idxList) {{
  layerGroup.clearLayers();
  const rute = DATA[activeTab].rute;
  const col  = activeTab === 'ga' ? '#2563eb' : '#16a34a';
  const seen = new Set();
  const bounds = [];

  idxList.forEach(idx => {{
    (rute[idx] || []).forEach(seg => {{
      const poly = seg.jalur;
      for (let i = 0; i < poly.length - 1; i++) {{
        const p1 = poly[i], p2 = poly[i+1];
        L.polyline([[p1[0],p1[1]],[p2[0],p2[1]]], {{
          color: elevColor((p1[2]+p2[2])/2), weight: 5, opacity: .85
        }}).addTo(layerGroup);
        bounds.push([p1[0], p1[1]]);
      }}
      [[poly[0], seg.asal_nama, seg.is_depot_asal],
       [poly[poly.length-1], seg.tujuan_nama, seg.is_depot_tujuan]
      ].forEach(([pt, nm, isDep]) => {{
        if (seen.has(nm)) return;
        seen.add(nm);
        L.circleMarker([pt[0], pt[1]], {{
          radius: isDep ? 9 : 6, color: 'white', weight: 2,
          fillColor: isDep ? '#f59e0b' : col, fillOpacity: 1
        }}).bindPopup(`<b>${{nm}}</b>${{pt[2] != null ? '<br>Elevasi: '+pt[2].toFixed(1)+'m' : ''}}`).addTo(layerGroup);
      }});
    }});
  }});
  if (bounds.length) map.fitBounds(bounds, {{padding:[30,30]}});
}}

function cekBanned() {{
  const seen = new Set();
  DATA[activeTab].rute.forEach(hari => hari.forEach(seg => {{
    seen.add(seg.asal_nama); seen.add(seg.tujuan_nama);
  }}));
  const el = document.getElementById('banned-notice');
  const selisih = NAMES_TOTAL - seen.size;
  if (selisih > 0) {{
    el.style.display = 'block';
    el.textContent   = `⚠️ ${{selisih}} destinasi di-banned (jarak PP melebihi batas harian).`;
  }} else {{
    el.style.display = 'none';
  }}
}}

function buildCards() {{
  const metrik = DATA[activeTab].metrik;
  const wrap   = document.getElementById('cards-wrap');
  wrap.innerHTML = '';

  const totalKm  = metrik.reduce((s, m) => s + m.jarak, 0);
  const avgFat   = metrik.reduce((s, m) => s + m.fatigue, 0) / metrik.length;
  document.getElementById('stat-hari').textContent    = metrik.length;
  document.getElementById('stat-km').textContent      = totalKm.toFixed(1);
  document.getElementById('stat-fatigue').textContent = avgFat.toFixed(1);

  metrik.forEach((m, idx) => {{
    const isSabtu = m.hari.toLowerCase().includes('sabtu');
    const card    = document.createElement('div');
    card.className = 'day-card';
    card.innerHTML = `
      <div class="card-top">
        <span class="day-badge ${{isSabtu ? 'sabtu' : 'minggu'}}">${{isSabtu ? 'SABTU' : 'MINGGU'}}</span>
        <span class="card-title">${{m.hari}}</span>
      </div>
      <div class="metrics">
        <div class="metric-box"><div class="mk">Jarak</div><div class="mv">${{m.jarak.toFixed(1)}} km</div></div>
        <div class="metric-box"><div class="mk">Fatigue</div><div class="mv">${{m.fatigue.toFixed(1)}}</div></div>
        <div class="metric-box"><div class="mk">Skor Jalan</div><div class="mv">${{m.jalan.toFixed(1)}}</div></div>
      </div>`;
    card.onclick = () => {{
      document.querySelectorAll('.day-card').forEach(c => c.classList.remove('active-ga','active-aco'));
      card.classList.add(`active-${{activeTab}}`);
      renderMap([idx]);
    }};
    wrap.appendChild(card);
  }});

  cekBanned();
}}

function tampilkanSemua() {{
  document.querySelectorAll('.day-card').forEach(c => c.classList.remove('active-ga','active-aco'));
  renderMap(DATA[activeTab].metrik.map((_, i) => i));
}}

function switchTab(tab) {{
  activeTab = tab;
  document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
  document.querySelector(`.tab-btn.${{tab}}`).classList.add('active');

  const badge = document.getElementById('algo-badge');
  badge.className   = tab;
  badge.textContent = tab === 'ga' ? 'Genetic Algorithm' : 'Ant Colony Optimization';

  const btnAll = document.getElementById('btn-all');
  btnAll.className = tab;

  document.getElementById('sidebar-title').textContent =
    tab === 'ga' ? 'Itinerary Multi-Hari — GA' : 'Itinerary Multi-Hari — ACO';

  buildCards();
  tampilkanSemua();
}}

buildCards();
tampilkanSemua();
</script>
</body>
</html>"""

    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"🎉 Dashboard selesai! Buka: {OUTPUT_PATH}")
    return OUTPUT_PATH


# ═══════════════════════════════════════════════════════════════════════
# MAIN — load data → pre-flight → tune+run ACO → tune+run GA → dashboard
# ═══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    try:
        # ── Load data ──────────────────────────────────────────────────
        matriks_jarak    = pd.read_csv(os.path.join(TARGET_FOLDER, 'distance_matrix_osmnx.csv'),           index_col=0)
        matriks_jalan    = pd.read_csv(os.path.join(TARGET_FOLDER, 'road_condition_matrix_osmnx.csv'),     index_col=0)
        matriks_elevasi  = pd.read_csv(os.path.join(TARGET_FOLDER, 'elevation_matrix_surabaya.csv'),       index_col=0)
        df_cluster       = pd.read_csv(os.path.join(TARGET_FOLDER, 'destinasi_clustered_real_distance.csv'))

        with open(os.path.join(TARGET_FOLDER, 'route_polyline_registry.json'), 'r') as f:
            route_registry = json.load(f)

        names      = matriks_jarak.index.tolist()
        dist_np    = matriks_jarak.to_numpy()
        road_np    = matriks_jalan.to_numpy()
        elev_np    = matriks_elevasi.to_numpy()
        cluster_np = df_cluster['Cluster_ID'].to_numpy()

        # ── Pre-flight check (satu kali, dipakai keduanya) ─────────────
        MAX_KM_USER, BANNED_NODES = preflight_check(dist_np, names)

        # ══════════════════════════════════════════
        # FASE ACO
        # ══════════════════════════════════════════
        print("\n" + "=" * 65)
        print("🐜  TUNING PARAMETER ACO (OPTUNA)")
        print("=" * 65)

        def objective_aco(trial):
            return run_aco(
                dist_np, elev_np, road_np, cluster_np,
                alpha       = trial.suggest_float('alpha', 0.1, 1.5),
                beta        = trial.suggest_float('beta', 2.0, 8.0),
                evaporation = trial.suggest_float('evaporation', 0.05, 0.5),
                max_km_user = MAX_KM_USER,
                banned_nodes= BANNED_NODES,
                num_ants=15, iterations=15
            )[0]

        study_aco = optuna.create_study(direction='minimize')
        study_aco.optimize(objective_aco, n_trials=20)
        best_aco = study_aco.best_params

        print(f"✅ ACO Tuning selesai | Skor: {round(study_aco.best_value, 2)}")
        print(f"   Parameter: {best_aco}")

        print("\n▶ Eksekusi final ACO...")
        score_aco, rute_f_aco, log_f_aco = run_aco(
            dist_np, elev_np, road_np, cluster_np,
            alpha       = best_aco['alpha'],
            beta        = best_aco['beta'],
            evaporation = best_aco['evaporation'],
            max_km_user = MAX_KM_USER,
            banned_nodes= BANNED_NODES,
            num_ants=40, iterations=50
        )

        print("\n" + "=" * 65)
        print("====== HASIL ACO ======")
        print(pd.DataFrame(log_f_aco).to_string(index=False))
        print(f"\nFitness ACO    : {round(score_aco, 2)}")
        print(f"Total Hari     : {len(rute_f_aco)} ({len(rute_f_aco)//2} akhir pekan)")

        pd.DataFrame(log_f_aco).to_csv(
            os.path.join(TARGET_FOLDER, 'itinerary_multi_weekend_aco.csv'), index=False
        )

        # ══════════════════════════════════════════
        # FASE GA
        # ══════════════════════════════════════════
        print("\n" + "=" * 65)
        print("🤖  TUNING PARAMETER GA (OPTUNA)")
        print("=" * 65)

        def objective_ga(trial):
            return run_ga(
                dist_np, elev_np, road_np, cluster_np,
                pop_size     = trial.suggest_int('pop_size', 80, 160, step=20),
                prob_cross   = trial.suggest_float('prob_crossover', 0.75, 0.95),
                prob_mut     = trial.suggest_float('prob_mutasi', 0.05, 0.20),
                tournament_k = trial.suggest_int('tournament_k', 4, 8),
                max_km_user  = MAX_KM_USER,
                banned_nodes = BANNED_NODES,
                iterations=30
            )[0]

        study_ga = optuna.create_study(direction='minimize')
        study_ga.optimize(objective_ga, n_trials=30)
        best_ga = study_ga.best_params

        print(f"✅ GA Tuning selesai | Skor: {round(study_ga.best_value, 2)}")
        print(f"   Parameter: {best_ga}")

        TOTAL_RUNS    = 10
        score_f_ga    = float('inf')
        rute_f_ga     = None
        log_f_ga      = None
        run_terbaik   = 0

        print(f"\n▶ Eksekusi final GA ({TOTAL_RUNS} runs)...")
        print("=" * 65)

        for id_run in range(1, TOTAL_RUNS + 1):
            sc, rt, lg = run_ga(
                dist_np, elev_np, road_np, cluster_np,
                pop_size     = int(best_ga['pop_size']),
                prob_cross   = best_ga['prob_crossover'],
                prob_mut     = best_ga['prob_mutasi'],
                tournament_k = int(best_ga['tournament_k']),
                max_km_user  = MAX_KM_USER,
                banned_nodes = BANNED_NODES,
                iterations=100
            )
            print(f"   [RUN {id_run:02d}/{TOTAL_RUNS}] Fitness: {sc:,.2f} | Trip: {len(rt)} hari")
            if sc < score_f_ga:
                score_f_ga = sc
                rute_f_ga  = rt
                log_f_ga   = lg
                run_terbaik = id_run

        print("\n" + "=" * 65)
        print("====== HASIL GA ======")
        print(pd.DataFrame(log_f_ga).to_string(index=False))
        print(f"\nFitness GA Global  : {round(score_f_ga, 2)}")
        print(f"Run terbaik        : [RUN {run_terbaik}]")
        print(f"Total Hari         : {len(rute_f_ga)} ({len(rute_f_ga)//2} akhir pekan)")

        pd.DataFrame(log_f_ga).to_csv(
            os.path.join(TARGET_FOLDER, 'itinerary_multi_weekend_ga.csv'), index=False
        )

        # ══════════════════════════════════════════
        # SATU DASHBOARD GABUNGAN
        # ══════════════════════════════════════════
        generate_dashboard_gabungan(
            rute_f_ga=rute_f_ga,   log_f_ga=log_f_ga,
            rute_f_aco=rute_f_aco, log_f_aco=log_f_aco,
            names=names,
            route_registry=route_registry,
            target_folder=TARGET_FOLDER,
        )

    except FileNotFoundError as e:
        print(f"[ERROR] File tidak ditemukan: {e}")
        print(f"Pastikan semua CSV & JSON ada di folder: {TARGET_FOLDER}")


🚲  PRE-FLIGHT SANITY CHECK: KAPASITAS USER
✅ Semua destinasi lolos uji kapasitas fisik harian!

🐜  TUNING PARAMETER ACO (OPTUNA)
✅ ACO Tuning selesai | Skor: 20959.04
   Parameter: {'alpha': 1.1148415703498877, 'beta': 5.934245020530545, 'evaporation': 0.3519454059856041}

▶ Eksekusi final ACO...

====== HASIL ACO ======
                    Hari/Trip  Jarak (Km)  Fatigue Index  Skor Jalan OSMnx  P_Fatigue  P_Cluster
Pekan 1 - Sabtu (Jalan Berat)       25.76         737.13             62.03    1105.69        0.0
  Pekan 1 - Minggu (Recovery)       25.48         406.56             38.20    3252.51      100.0
Pekan 2 - Sabtu (Jalan Berat)       25.70        7833.11             30.08   11749.66       50.0
  Pekan 2 - Minggu (Recovery)       18.63         628.91             13.44    5031.31        0.0

Fitness ACO    : 23083.63
Total Hari     : 4 (2 akhir pekan)

🤖  TUNING PARAMETER GA (OPTUNA)
✅ GA Tuning selesai | Skor: 36770.23
   Parameter: {'pop_size': 100, 'prob_crossover': 0.8403811